# LangChain + LangGraph Lab: Documentation RAG Agent

This notebook recreates the Evidently documentation RAG agent using **LangChain** and **LangGraph**.

- **LangChain** provides the OpenAI chat model, tool abstraction, and message classes.
- **LangGraph** provides the explicit agent workflow: shared state, nodes, conditional edges, guardrails, and checkpointing.

## 1. Install dependencies

Run this terminal command once from the notebook folder:

```bash
uv add langchain langchain-openai langgraph gitsource minsearch
```

Or run it directly from the notebook:

```bash
!uv add langchain langchain-openai langgraph gitsource minsearch
```

These are the dependencies used by the earlier notebooks:

- `gitsource`: reads the Evidently documentation GitHub repository.
- `minsearch`: provides the keyword search index.
- `langchain` and `langchain-openai`: provide the model and LangChain tool integration.
- `langgraph`: provides the graph runtime.

## 2. Set your OpenAI API key

Set this in your terminal before starting Jupyter:

```bash
export OPENAI_API_KEY="your-openai-api-key"
```

Or, for a temporary notebook-only setting:

```python
import os
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
```

Avoid committing API keys to source control.

### Short production reminder

For production, replace print-based callbacks with structured logs or tracing events. Include request/run IDs, tool name, latency, status, and redacted argument summaries. Do not log sensitive content or full tool arguments by default. Keep callbacks observational; enforce authorization and validation inside the tool execution boundary.


In [1]:
# Install dependencies if needed.
# In a uv-managed project, this updates pyproject.toml and uv.lock.

!uv add langchain langchain-openai langgraph gitsource minsearch

Resolved 166 packages in 65ms
Checked 162 packages in 2.29s


## 3. Build the documentation knowledge base

This copies the setup from the earlier notebooks.

We read the Evidently documentation repository, parse Markdown and MDX files, chunk them, and index the chunks with `minsearch`.

Important: this version of `gitsource` uses:

```python
from gitsource import GithubRepositoryDataReader, chunkdocuments
```

and calls:

```python
chunkdocuments(files, size=3000, step=1500)
```



In [3]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = list(reader.read())

# Convert RawRepositoryFile objects into dictionaries.
parsed_docs = [doc.parse() for doc in files]

chunked_docs = chunk_documents(
    parsed_docs,
    size=3000,
    step=1500,
)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"],
)

index.fit(chunked_docs)

print(f"Files: {len(files)}")
print(f"Chunks: {len(chunked_docs)}")

Files: 95
Chunks: 382


## 4. Create the shared search tools

These functions match the interface expected by the framework notebooks:

```python
search_tools.search(query)
search_tools.get_file(filename)
```

`get_file()` combines all indexed chunks belonging to one filename, ordered by their starting position.

In [4]:
from typing import List


class SearchTools:
    """Documentation search tools backed by the MinSearch index."""

    def __init__(self, files, chunked_docs, index):
        self.files = files
        self.chunked_docs = chunked_docs
        self.index = index

    def search(self, query: str, num_results: int = 5) -> List[dict]:
        """Search the documentation index for relevant chunks."""
        return self.index.search(query, num_results=num_results)

    def get_file(self, filename: str) -> str:
        """Return the full text of a documentation file by filename."""
        matching_chunks = [
            chunk
            for chunk in self.chunked_docs
            if chunk.get("filename") == filename
        ]

        if not matching_chunks:
            raise FileNotFoundError(
                f"Documentation file not found in index: {filename}"
            )

        ordered_chunks = sorted(
            matching_chunks,
            key=lambda chunk: chunk.get("start", 0),
        )

        return "\n\n".join(chunk["content"] for chunk in ordered_chunks)


search_tools = SearchTools(
    files=files,
    chunked_docs=chunked_docs,
    index=index,
)

results = search_tools.search("dashboard")
print(results[0]["filename"])

docs/platform/dashboard_add_panels.mdx


## 5. Define agent instructions

This prompt follows the same pattern as the earlier documentation assistant notebooks.

In [5]:
AGENT_INSTRUCTIONS = """
You are a helpful documentation assistant for Evidently.

Use the available tools to research the documentation before answering.

When research is needed:
1. Start with one focused search.
2. Analyze the results and perform targeted follow-up searches if needed.
3. Use get_file when a search result identifies a particularly relevant filename.
4. Synthesize the evidence into a clear final answer.

Answer only from the documentation returned by the tools.
If the documentation does not contain the answer, say that you do not know.
Include relevant code examples when they are available.
""".strip()

print(AGENT_INSTRUCTIONS)

You are a helpful documentation assistant for Evidently.

Use the available tools to research the documentation before answering.

When research is needed:
1. Start with one focused search.
2. Analyze the results and perform targeted follow-up searches if needed.
3. Use get_file when a search result identifies a particularly relevant filename.
4. Synthesize the evidence into a clear final answer.

Answer only from the documentation returned by the tools.
If the documentation does not contain the answer, say that you do not know.
Include relevant code examples when they are available.


## 6. Create the LangChain model and tools

`ChatOpenAI` is the LangChain OpenAI chat-model client.

We wrap the existing `search_tools` methods with `StructuredTool.from_function()`, so we do not need to redefine their docstrings.

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import StructuredTool

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

tools = [
    StructuredTool.from_function(search_tools.search),
    StructuredTool.from_function(search_tools.get_file),
]

tool_by_name = {tool.name: tool for tool in tools}

print([tool.name for tool in tools])

['search', 'get_file']


## 7. Define the LangGraph state

LangGraph passes one shared state object between nodes.

- `messages` accumulates the conversation.
- `retrieved_context` collects search results.
- `tool_call_count` supports a guardrail.
- `request_id` correlates logs for one run.

In [7]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain_core.messages import AnyMessage


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    retrieved_context: Annotated[list[str], operator.add]
    tool_call_count: int
    request_id: str

## 8. Create structured logging

Production-style agents should emit structured events rather than relying on `print()`.

We log:

- Request ID
- Tool name
- Redacted argument summary
- Latency
- Status
- Tool-call count
- Number of retrieved context chunks

In [ ]:
import time
import logging
import uuid

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
)

logger = logging.getLogger("langgraph_docs_agent")


def summarize_args(args, limit: int = 120) -> str:
    """Return a short, redacted summary suitable for logs."""
    text = str(args)
    return text if len(text) <= limit else text[:limit] + "..."

## 9. Define graph nodes

The `agent` node asks the model what to do next.

The `tools` node executes requested tools, records telemetry, and returns results to the model.

The `respond` node produces the final answer.

In [ ]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

SYSTEM_PROMPT = AGENT_INSTRUCTIONS


def agent_node(state: AgentState) -> dict:
    """Ask the model whether to call a tool or produce a final answer."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]

    llm_with_tools = llm.bind_tools(tools)
    response = llm_with_tools.invoke(messages)

    return {"messages": [response]}


def tools_node(state: AgentState) -> dict:
    """Execute tool calls requested by the model."""
    last_message = state["messages"][-1]

    tool_messages = []
    retrieved_context = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        started = time.perf_counter()

        try:
            if tool_name not in tool_by_name:
                raise ValueError(f"Unknown tool: {tool_name}")

            tool_result = tool_by_name[tool_name].invoke(tool_args)
            status = "success"
            content = str(tool_result)

            if tool_name == "search":
                retrieved_context.append(content)

        except Exception as exc:
            status = "error"
            content = f"Tool failed: {type(exc).__name__}: {exc}"

        latency_ms = round((time.perf_counter() - started) * 1000, 2)

        logger.info(
            "agent_tool_completed",
            extra={
                "event": "tool_completed",
                "request_id": state["request_id"],
                "tool_name": tool_name,
                "args_summary": summarize_args(tool_args),
                "latency_ms": latency_ms,
                "status": status,
            },
        )

        tool_messages.append(
            ToolMessage(
                content=content,
                tool_call_id=tool_call_id,
                name=tool_name,
            )
        )

    return {
        "messages": tool_messages,
        "retrieved_context": retrieved_context,
        "tool_call_count": state["tool_call_count"] + len(tool_messages),
    }


def respond_node(state: AgentState) -> dict:
    """Generate the final response from the accumulated conversation."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]

    response = llm.invoke(messages)

    logger.info(
        "agent_response_generated",
        extra={
            "event": "response_generated",
            "request_id": state["request_id"],
            "tool_call_count": state["tool_call_count"],
            "context_chunks": len(state["retrieved_context"]),
        },
    )

    return {"messages": [response]}

## 10. Add routing and a guardrail

The router decides what happens after the `agent` node.

If the model requested tools, go to `tools`. Otherwise, go to `respond`.

If the agent has already made too many tool calls, stop the loop and answer from what has been retrieved.

In [ ]:
MAX_TOOL_CALLS = 6


def route_after_agent(state: AgentState) -> Literal["tools", "respond"]:
    last_message = state["messages"][-1]

    if state["tool_call_count"] >= MAX_TOOL_CALLS:
        logger.warning(
            "tool_call_limit_reached",
            extra={
                "request_id": state["request_id"],
                "tool_call_count": state["tool_call_count"],
                "limit": MAX_TOOL_CALLS,
            },
        )
        return "respond"

    if last_message.tool_calls:
        return "tools"

    return "respond"

## 11. Build and visualize the graph

The graph makes the agent loop explicit:

```text
START
  ↓
agent
  ↓
route_after_agent
  ├── tools → agent
  └── respond → END
```



In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)
builder.add_node("respond", respond_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {
        "tools": "tools",
        "respond": "respond",
    },
)

builder.add_edge("tools", "agent")
builder.add_edge("respond", END)

graph = builder.compile()

graph

## 12. Run the graph

Create an initial state with the user question, then invoke the graph.

The final assistant message is the last item in `result["messages"]`.

In [ ]:
query = "how do I use evidently to monitor my machine learning models?"

initial_state = {
    "messages": [HumanMessage(content=query)],
    "retrieved_context": [],
    "tool_call_count": 0,
    "request_id": str(uuid.uuid4()),
}

result = graph.invoke(initial_state)

print(result["messages"][-1].content)

## 13. Inspect the execution

This shows the user question, tool calls, tool results, and final answer.

In [ ]:
def print_graph_messages(messages):
    for message in messages:
        if isinstance(message, HumanMessage):
            print(f"User: {message.content[:200]}")

        elif isinstance(message, AIMessage):
            if message.content:
                print(f"Assistant: {message.content[:300]}")

            if message.tool_calls:
                for tool_call in message.tool_calls:
                    print(
                        f"Tool call: {tool_call['name']}({tool_call['args']})"
                    )

        elif isinstance(message, ToolMessage):
            print(f"Tool result: {str(message.content)[:200]}...")

        print()


print_graph_messages(result["messages"])

In [ ]:
print(f"Tool calls executed: {result['tool_call_count']}")
print(f"Context chunks collected: {len(result['retrieved_context'])}")
print(f"Request ID: {result['request_id']}")

## 14. Continue the conversation

LangGraph state is explicit. To continue without a checkpointer, carry the prior `messages` into the next invocation.

In [ ]:
follow_up = "show me the code"

follow_up_state = {
    "messages": result["messages"] + [HumanMessage(content=follow_up)],
    "retrieved_context": [],
    "tool_call_count": 0,
    "request_id": str(uuid.uuid4()),
}

follow_up_result = graph.invoke(follow_up_state)

print(follow_up_result["messages"][-1].content)

In [ ]:
print_graph_messages(follow_up_result["messages"])

## 15. Add checkpoint-based memory

`MemorySaver` stores graph state after each step. It is appropriate for notebooks and demos.

For production, use a durable checkpointer, such as SQLite or Postgres, and isolate conversations using a `thread_id`.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

persistent_graph = builder.compile(checkpointer=checkpointer)

thread_config = {
    "configurable": {
        "thread_id": "docs-conversation-001"
    }
}

persistent_result = persistent_graph.invoke(
    {
        "messages": [HumanMessage(content=query)],
        "retrieved_context": [],
        "tool_call_count": 0,
        "request_id": str(uuid.uuid4()),
    },
    config=thread_config,
)

print(persistent_result["messages"][-1].content)

In [ ]:
# The same thread_id restores prior conversation state automatically.
persistent_follow_up = persistent_graph.invoke(
    {
        "messages": [HumanMessage(content="show me the code")],
    },
    config=thread_config,
)

print(persistent_follow_up["messages"][-1].content)

In [ ]:
snapshot = persistent_graph.get_state(thread_config)

print("Next node:", snapshot.next)
print("Stored messages:", len(snapshot.values["messages"]))
print("Tool calls:", snapshot.values["tool_call_count"])

## 16. Review run telemetry

The graph nodes already emit structured log events. This cell summarizes the two manual runs in a DataFrame.

In [ ]:
import pandas as pd

run_summary = pd.DataFrame(
    [
        {
            "request_id": result["request_id"],
            "event": "run_completed",
            "tool_call_count": result["tool_call_count"],
            "context_chunks": len(result["retrieved_context"]),
        },
        {
            "request_id": follow_up_result["request_id"],
            "event": "run_completed",
            "tool_call_count": follow_up_result["tool_call_count"],
            "context_chunks": len(follow_up_result["retrieved_context"]),
        },
    ]
)

run_summary

## What you learned

- LangChain supplies the model, tools, and message classes.
- LangGraph supplies explicit state, nodes, edges, routing, and checkpointing.
- Tool execution can be a visible graph node rather than an opaque agent loop.
- Conditional edges make agent behavior easier to inspect and control.
- Checkpointers enable resumable multi-turn conversations.
- Production agents need structured observability, guardrails, authorization, and validation.

## Client production checklist

- Authenticate users and isolate conversations by tenant and thread.
- Use a durable checkpointer for real conversations.
- Start with a small set of read-only tools.
- Validate inputs and check permissions inside tool functions.
- Add tool-call limits, timeouts, retries, and cost controls.
- Log request/thread IDs, tool names, latency, status, and redacted arguments.
- Build an evaluation set before exposing the agent to clients.
- Test normal, invalid-input, unauthorized, and upstream-failure paths.